<a href="https://colab.research.google.com/github/peculab/Studies-in-Network-aided-Learning-Systems/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("test")

test


In [8]:
import re
import json
import time
import random
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from typing import Optional, List, Dict, Tuple

import requests
from bs4 import BeautifulSoup

PTT_BASE = "https://www.ptt.cc"
BOARD = "Stock"
OUTFILE = "ptt_finance_last10days.jsonl"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ptt-crawler/1.1)",
    "Accept-Language": "zh-TW,zh;q=0.9,en;q=0.8",
}

TZ_TW = timezone(timedelta(hours=8))

In [9]:
@dataclass
class Push:
    tag: str
    user: str
    content: str
    time: str


@dataclass
class Post:
    url: str
    title: Optional[str]
    author: Optional[str]
    datetime_iso: Optional[str]
    board: str
    content: Optional[str]
    ip: Optional[str]
    pushes: List[Push]
    deleted: bool


def polite_sleep():
    time.sleep(random.uniform(0.35, 0.9))


def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def get_soup(session: requests.Session, url: str) -> BeautifulSoup:
    r = session.get(url, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def pass_over18(session: requests.Session, board: str):
    """
    走一次官方 over18 流程：POST /ask/over18
    """
    target = f"/bbs/{board}/index.html"
    url = f"{PTT_BASE}/ask/over18"
    payload = {"from": target, "yes": "yes"}
    r = session.post(url, data=payload, timeout=20)
    r.raise_for_status()
    # 成功後會在 cookie 裡有 over18=1
    if session.cookies.get("over18") != "1":
        # 有些情況 cookie domain/path 不同，這裡保底再手動補一次
        session.cookies.set("over18", "1", domain=".ptt.cc", path="/")


def get_latest_index_url(session: requests.Session) -> str:
    url = f"{PTT_BASE}/bbs/{BOARD}/index.html"
    r = session.get(url, timeout=20, allow_redirects=True)
    r.raise_for_status()
    return r.url


def parse_index_posts(soup: BeautifulSoup) -> List[Tuple[str, str]]:
    items = []
    for div in soup.select("div.r-ent"):
        title_div = div.select_one("div.title")
        title_text = title_div.get_text(strip=True) if title_div else ""
        a = div.select_one("div.title a")
        if a and a.get("href"):
            items.append((title_text, PTT_BASE + a["href"]))
        else:
            items.append((title_text, ""))  # deleted
    return items


def parse_prev_page_url(soup: BeautifulSoup) -> Optional[str]:
    for a in soup.select("div.btn-group.btn-group-paging a"):
        if a.get_text(strip=True) == "‹ 上頁" and a.get("href"):
            return PTT_BASE + a["href"]
    return None


def parse_article_datetime_tw(soup: BeautifulSoup) -> Optional[datetime]:
    for m in soup.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        val = m.select_one("span.article-meta-value")
        if not tag or not val:
            continue
        if tag.get_text(strip=True) == "時間":
            t = val.get_text(strip=True)
            try:
                dt = datetime.strptime(t, "%a %b %d %H:%M:%S %Y")
                return dt.replace(tzinfo=TZ_TW)
            except Exception:
                return None
    return None


def parse_article_meta(soup: BeautifulSoup) -> Dict[str, Optional[str]]:
    author = title = ip = None

    for m in soup.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        val = m.select_one("span.article-meta-value")
        if not tag or not val:
            continue
        k = tag.get_text(strip=True)
        v = val.get_text(strip=True)
        if k == "作者":
            author = v
        elif k == "標題":
            title = v

    main = soup.select_one("div#main-content")
    if main:
        text = main.get_text("\n", strip=False)
        m = re.search(r"來自:\s*([0-9]{1,3}(?:\.[0-9]{1,3}){3})", text)
        if m:
            ip = m.group(1)

    return {"author": author, "title": title, "ip": ip}


def extract_main_content(soup: BeautifulSoup) -> str:
    main = soup.select_one("div#main-content")
    if not main:
        return ""

    for m in main.select("div.article-metaline"):
        m.decompose()
    for m in main.select("div.article-metaline-right"):
        m.decompose()
    for p in main.select("div.push"):
        p.decompose()

    content = main.get_text("\n", strip=False)
    content = content.split("※ 發信站:")[0]

    lines = [ln.rstrip() for ln in content.splitlines()]
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines).strip()


def parse_pushes(soup: BeautifulSoup) -> List[Push]:
    pushes: List[Push] = []
    for p in soup.select("div.push"):
        tag = p.select_one("span.push-tag")
        user = p.select_one("span.push-userid")
        content = p.select_one("span.push-content")
        push_time = p.select_one("span.push-ipdatetime")
        pushes.append(
            Push(
                tag=tag.get_text(strip=True) if tag else "",
                user=user.get_text(strip=True) if user else "",
                content=(content.get_text(strip=True).lstrip(":") if content else ""),
                time=push_time.get_text(strip=True) if push_time else "",
            )
        )
    return pushes


def fetch_post(session: requests.Session, url: str, fallback_title: Optional[str] = None) -> Post:
    if not url:
        return Post(
            url="",
            title=fallback_title,
            author=None,
            datetime_iso=None,
            board=BOARD,
            content=None,
            ip=None,
            pushes=[],
            deleted=True,
        )

    soup = get_soup(session, url)
    dt = parse_article_datetime_tw(soup)
    meta = parse_article_meta(soup)
    content = extract_main_content(soup)
    pushes = parse_pushes(soup)

    return Post(
        url=url,
        title=meta["title"] or fallback_title,
        author=meta["author"],
        datetime_iso=dt.isoformat() if dt else None,
        board=BOARD,
        content=content,
        ip=meta["ip"],
        pushes=pushes,
        deleted=False,
    )


def append_jsonl(path: str, post: Post):
    with open(path, "a", encoding="utf-8") as f:
        d = asdict(post)
        d["pushes"] = [asdict(x) for x in post.pushes]
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
        f.flush()


def crawl_stream_last_n_days(days: int = 10):
    session = make_session()

    # ✅ 先過 over18 gate（就算 Finance 不一定需要，也不會有副作用）
    pass_over18(session, BOARD)

    latest = get_latest_index_url(session)
    cutoff = datetime.now(TZ_TW) - timedelta(days=days)

    print(f"Cutoff (TW): {cutoff.isoformat()}")
    print(f"Start index: {latest}")
    print(f"Output: {OUTFILE}")
    print("-" * 80)

    # 清空輸出
    open(OUTFILE, "w", encoding="utf-8").close()

    idx_url = latest
    count = 0

    while True:
        soup = get_soup(session, idx_url)
        entries = parse_index_posts(soup)

        # 🔎 Debug：如果抓不到 r-ent，印出頁面標題讓你一眼知道被擋
        if not entries:
            page_title = soup.title.get_text(strip=True) if soup.title else "(no title)"
            print(f"[WARN] No entries found on {idx_url}. Page title: {page_title}")
            # 也印出一點文字線索
            snippet = soup.get_text("\n", strip=True)[:200]
            print(f"[WARN] Page text snippet: {snippet}")
            return

        for title_text, post_url in entries:
            polite_sleep()
            post = fetch_post(session, post_url, fallback_title=title_text or None)

            # cutoff 判斷
            if post.datetime_iso:
                dt = datetime.fromisoformat(post.datetime_iso)
                if dt < cutoff:
                    print("-" * 80)
                    print(f"Reached older than cutoff. Stop at {post.datetime_iso}.")
                    print(f"Total saved: {count}")
                    return

            count += 1
            append_jsonl(OUTFILE, post)

            # 邊抓邊印（輕量）
            if post.deleted:
                print(f"[{count:05d}] (deleted) {post.title or ''}")
            else:
                print(f"[{count:05d}] {post.datetime_iso} | {post.title} | {post.url}")

        prev_url = parse_prev_page_url(soup)
        if not prev_url:
            print(f"End reached. Total saved: {count}")
            return
        idx_url = prev_url


if __name__ == "__main__":
    crawl_stream_last_n_days(days=10)

Cutoff (TW): 2026-02-13T10:35:35.829448+08:00
Start index: https://www.ptt.cc/bbs/Stock/index.html
Output: ptt_finance_last10days.jsonl
--------------------------------------------------------------------------------
[00001] (deleted) (已被BlueBird5566刪除) <hitguy> 4-4
[00002] 2026-02-23T08:30:01+08:00 | [閒聊] 2026/02/23 盤中閒聊 | https://www.ptt.cc/bbs/Stock/M.1771806605.A.201.html
[00003] 2026-02-23T08:42:43+08:00 | [新聞] 韓國綜合股價指數（KOSPI）突破5900點 | https://www.ptt.cc/bbs/Stock/M.1771807367.A.C4A.html
[00004] 2026-02-23T08:47:54+08:00 | [新聞] 輝達、OpenAI千億合作案縮水！金額或僅 | https://www.ptt.cc/bbs/Stock/M.1771807676.A.047.html
[00005] 2026-02-23T09:32:07+08:00 | [新聞] 擴產也來不及！全球記憶體荒將成AI競爭 | https://www.ptt.cc/bbs/Stock/M.1771810329.A.DCA.html
[00006] 2026-02-23T09:58:48+08:00 | [請益] AI寫程式交易碼的穩定性？ | https://www.ptt.cc/bbs/Stock/M.1771811930.A.0B6.html
[00007] 2026-02-23T10:21:21+08:00 | [新聞] 聯發科蔡力行示警AI恐撞「電力牆」 拋10 | https://www.ptt.cc/bbs/Stock/M.1771813283.A.BAE.html
----------------------------------------